# Tests for various Ks

## Imports

In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import pandas as pd
import numpy as np
import sys
import warnings
import time

project_dir = Path.cwd().parent

sys.path.insert(0, str(Path.cwd().parent)) # import src/
warnings.filterwarnings("ignore")

pd.set_option("display.width", 220, "display.max_columns", 30)

from src import (NestedCVRegressor, default_models,
                           default_param_spaces, MRMRSelector, run_per_family,
                           alt_param_spaces, build_xy, load_anndata)

HVG_DATA_DIR = project_dir / 'data' / 'HVGs'

## Dataset

In [ ]:
FAMILIES = {
    "KenyonCells":  (HVG_DATA_DIR / "Kenyon_cells_train_hvg.h5ad",  HVG_DATA_DIR / "Kenyon_cells_test_hng.h5ad"),
    "OpticLobe":    (HVG_DATA_DIR / "Optic_lobe_train_hvg.h5ad",    HVG_DATA_DIR / "Optic_lobe_test_hng.h5ad"),
    "Monoaminergic":(HVG_DATA_DIR / "Monoaminergic_train_hvg.h5ad",HVG_DATA_DIR / "Monoaminergic_test_hng.h5ad"),
    "Glia":         (HVG_DATA_DIR / "Glia_train_hvg.h5ad",         HVG_DATA_DIR / "Glia_test_hng.h5ad"),
    "Peptidergic":  (HVG_DATA_DIR / "Peptidergic_train_hvg.h5ad",  HVG_DATA_DIR / "Peptidergic_test_hng.h5ad"),
    "Clock":        (HVG_DATA_DIR / "Clock_train_hvg.h5ad",        HVG_DATA_DIR / "Clock_test_hng.h5ad"),
}
FAMILIES = {f: p for f, p in FAMILIES.items() if p[0].exists() and p[1].exists()}

## Dataset

In [7]:
X, y, _ = build_xy(load_anndata(FAMILIES["Clock"][0]), extra_obs=("nUMI",))

K_VALUES = [50, 100, 200, 300, 500, 2000]

def variable_K_tests(X, y, fs_mode, K_values=K_VALUES, tune=False,
            n_rounds=1, n_outer=5, n_inner=3, n_trials=15,
            inner_scoring="neg_root_mean_squared_error"):
    rows = []
    for K in K_values:
        sel = None if K >= X.shape[1] else MRMRSelector(k=K, relevance="f")
        reg = NestedCVRegressor(
            default_models(),
            alt_param_spaces() if tune else None,
            feature_selector=sel, fs_mode=fs_mode, k_safety_cap=False,
            n_rounds=n_rounds, n_outer=n_outer, n_inner=n_inner,
            n_trials=n_trials, inner_scoring=inner_scoring,
        )
        t0 = time.time(); res = reg.run(X, y, tune=tune, verbose=0); dt = time.time() - t0
        for _, r in res.summary.iterrows():
            rows.append({"K": K, "model": r["model"],
                         "MAE": round(r["MAE_median"], 3), "RMSE": round(r["RMSE_median"], 3),
                         "R2": round(r["R2_median"], 3), "Spearman": round(r["Spearman_median"], 3)})
        print(f"  [{fs_mode}] K={K:>4}: {dt:.0f}s", flush=True)
    return pd.DataFrame(rows)

## Tests

### Tuned outer

In [9]:
exp_outer = variable_K_tests(X, y, fs_mode="outer", tune=True, n_rounds=1, n_outer=5, n_inner=3, n_trials=15)

print("\nMAE by model x K  (outer):")
print(exp_outer.pivot(index="model", columns="K", values="MAE"))

  [outer] K=  50: 166s
  [outer] K= 100: 240s
  [outer] K= 200: 684s
  [outer] K= 300: 778s
  [outer] K= 500: 1150s
  [outer] K=2000: 1572s

MAE by model x K  (outer):
K                      50     100    200     300    500    2000
model                                                          
Dummy                 9.728  9.728  9.728   9.728  9.728  9.728
ElasticNet            6.904  7.229  7.269   7.453  7.002  6.908
LinearRegression      7.583  8.020  9.868  16.495  9.102  6.264
RandomForest          5.344  5.296  5.390   5.556  5.651  5.891
RandomForest_authors  6.407  6.818  6.891   6.916  6.824  7.218
SVR                   6.857  7.782  8.104   8.067  7.950  7.955
XGBoost               5.576  5.311  5.590   5.717  5.549  5.894


In [10]:
print("\nRMSE by model x K  (outer):")
print(exp_outer.pivot(index="model", columns="K", values="RMSE"))


RMSE by model x K  (outer):
K                       50      100     200     300     500     2000
model                                                               
Dummy                 12.991  12.991  12.991  12.991  12.991  12.991
ElasticNet            10.502  10.313  10.254  10.294  10.018   9.484
LinearRegression      11.287  11.298  13.261  21.137  11.981   8.474
RandomForest           8.541   8.744   8.952   8.805   8.798   9.001
RandomForest_authors  10.411  10.512  10.672  10.667  10.318  10.716
SVR                   10.157  10.905  10.848  10.773  10.852  10.799
XGBoost                8.782   8.304   8.789   8.732   8.810   8.809


### Tuned inner

In [8]:
exp_inner = variable_K_tests(X, y, fs_mode="inner", tune=True, n_rounds=1, n_outer=5, n_inner=3, n_trials=15)

print("\nMAE by model x K  (inner):")
print(exp_inner.pivot(index="model", columns="K", values="MAE"))

  [inner] K=  50: 186s
  [inner] K= 100: 217s
  [inner] K= 200: 250s
  [inner] K= 300: 280s
  [inner] K= 500: 325s
  [inner] K=2000: 888s

MAE by model x K  (inner):
K                      50     100    200     300    500    2000
model                                                          
Dummy                 9.728  9.728  9.728   9.728  9.728  9.728
ElasticNet            6.798  6.863  6.909   6.669  6.643  6.908
LinearRegression      7.583  8.020  9.868  16.495  9.102  6.264
RandomForest          5.371  5.342  5.463   5.587  5.651  5.912
RandomForest_authors  6.407  6.818  6.891   6.916  6.824  7.218
SVR                   6.830  7.418  7.911   7.339  7.704  7.955
XGBoost               5.306  5.452  5.590   5.354  5.442  5.894


In [11]:
print("\nRMSE by model x K  (inner):")
print(exp_inner.pivot(index="model", columns="K", values="RMSE"))


RMSE by model x K  (inner):
K                       50      100     200     300     500     2000
model                                                               
Dummy                 12.991  12.991  12.991  12.991  12.991  12.991
ElasticNet             9.984   9.833   9.763   9.460   9.487   9.484
LinearRegression      11.287  11.298  13.261  21.137  11.981   8.474
RandomForest           8.500   8.667   8.830   8.805   8.689   9.027
RandomForest_authors  10.411  10.512  10.672  10.667  10.318  10.716
SVR                   10.354  10.963  10.848  11.147  11.033  10.799
XGBoost                8.892   8.756   8.789   8.812   8.883   8.712


## Inner tune 5 iters

In [24]:
K_VALUES_FEW = [50, 100, 150]

def few_Ks(X, y, fs_mode, K_values=K_VALUES_FEW, tune=False,
            n_rounds=5, n_outer=5, n_inner=3, n_trials=15,
            inner_scoring="neg_root_mean_squared_error", relevance="f"):
    rows = []
    for K in K_values:
        sel = None if K >= X.shape[1] else MRMRSelector(k=K, relevance=relevance)
        reg = NestedCVRegressor(
            {"RandomForest": default_models()["RandomForest"],
             "XGBoost": default_models()["XGBoost"]},
            alt_param_spaces() if tune else None,
            feature_selector=sel, fs_mode=fs_mode, k_safety_cap=False,
            n_rounds=n_rounds, n_outer=n_outer, n_inner=n_inner,
            n_trials=n_trials, inner_scoring=inner_scoring,
        )
        t0 = time.time(); res = reg.run(X, y, tune=tune, verbose=0); dt = time.time() - t0
        for _, r in res.summary.iterrows():
            rows.append({"K": K, "model": r["model"],
                         "MAE": round(r["MAE_median"], 3), "RMSE": round(r["RMSE_median"], 3),
                         "R2": round(r["R2_median"], 3), "Spearman": round(r["Spearman_median"], 3)})
        print(f"  [{fs_mode}] K={K:>4}: {dt:.0f}s", flush=True)
    return pd.DataFrame(rows)

In [21]:
exp_inner_few = few_Ks(X, y, fs_mode="inner", tune=True, n_rounds=1, n_outer=5, n_inner=3, n_trials=15)


  [inner] K=  50: 166s
  [inner] K= 100: 221s
  [inner] K= 150: 252s


In [22]:
print("\nMAE by model x K  (inner):")
print(exp_inner_few.pivot(index="model", columns="K", values="MAE"))
print("\nRMSE by model x K  (inner):")
print(exp_inner_few.pivot(index="model", columns="K", values="RMSE"))


MAE by model x K  (inner):
K               50     100    150
model                            
RandomForest  5.371  5.342  5.525
XGBoost       5.306  5.452  5.390

RMSE by model x K  (inner):
K               50     100    150
model                            
RandomForest  8.500  8.667  8.819
XGBoost       8.892  8.756  8.617


In [25]:
exp_inner_few_mi = few_Ks(X, y, fs_mode="inner", tune=True, n_rounds=1, n_outer=5, n_inner=3, n_trials=15, relevance="mi")


  [inner] K=  50: 293s
  [inner] K= 100: 674s
  [inner] K= 150: 1014s


In [26]:
print("\nMAE by model x K  (inner):")
print(exp_inner_few_mi.pivot(index="model", columns="K", values="MAE"))
print("\nRMSE by model x K  (inner):")
print(exp_inner_few_mi.pivot(index="model", columns="K", values="RMSE"))


MAE by model x K  (inner):
K               50     100    150
model                            
RandomForest  6.234  5.444  5.478
XGBoost       5.997  5.816  5.918

RMSE by model x K  (inner):
K               50     100    150
model                            
RandomForest  9.626  8.565  8.733
XGBoost       9.470  9.017  9.098
